# 🎙️ Sentence Bank TTS Generator
**Generates sentence-level and word-level TTS audio for 16 sentence banks (8 English + 8 Urdu)**

### Before Running:
1. Upload all 16 JSON files to Colab (Files panel → Upload)
2. Run cells in order
3. Download `sentence_bank_audio_assets.zip` at the end

### File Naming Convention:
- Sentence: `audio_assets/{language}/{test_type}/sentence/{id}.wav`
- Word: `audio_assets/{language}/{test_type}/word/{id}_w{idx}_{word}.wav`

### Settings (same as original TTS generator):
- **Model**: Coqui XTTS v2 — Speaker: Ana Florence
- **English**: `language="en"` | **Urdu**: `language="hi"` (Hindi mode)
- `temperature=0.65, top_k=150, top_p=0.80, repetition_penalty=1.5, speed=0.95`
- Post-processing: 250ms trim, -1dB normalize, 80Hz HPF, 50ms fade


In [ ]:
# ============================================================
# STEP 1: Environment Setup (Python 3.10 venv — bypasses Colab 3.12 crash)
# ============================================================
print("1️⃣ Setting up Python 3.10 environment (takes ~2 mins)...")
!sudo apt-get update -y > /dev/null
!sudo apt-get install software-properties-common -y > /dev/null
!sudo add-apt-repository ppa:deadsnakes/ppa -y > /dev/null
!sudo apt-get install python3.10 python3.10-venv python3.10-distutils -y > /dev/null

!python3.10 -m venv /content/tts-env
!/content/tts-env/bin/pip install -q setuptools wheel cython numpy==1.25.2
!/content/tts-env/bin/pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!/content/tts-env/bin/pip install -q TTS==0.22.0 transformers==4.37.2 librosa soundfile scipy

print("✅ Environment ready!")


In [ ]:
# ============================================================
# STEP 2: Verify JSON Files Are Uploaded
# ============================================================
import os

ENGLISH_FILES = [
    "1_blocks.json",
    "2_prolongation.json",
    "3_repetitions.json",
    "4_velar_fronting.json",
    "5_stopping.json",
    "6_gliding.json",
    "7_cluster_reduction.json",
    "8_epenthesis.json",
]

URDU_FILES = [
    "urdu_1_blocks.json",
    "urdu_2_prolongation.json",
    "urdu_3_repetitions.json",
    "urdu_4_velar_fronting.json",
    "urdu_5_stopping.json",
    "urdu_6_gliding.json",
    "urdu_7_cluster_reduction.json",
    "urdu_8_epenthesis.json",
]

print("2️⃣ Checking uploaded files...\n")
missing = []
for fname in ENGLISH_FILES + URDU_FILES:
    path = f"/content/{fname}"
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"  ✅ {fname}  ({size:.1f} KB)")
    else:
        print(f"  ❌ {fname}  — MISSING")
        missing.append(fname)

if missing:
    print(f"\n⚠️  Please upload the missing {len(missing)} file(s) before continuing.")
else:
    print(f"\n✅ All 16 JSON files found! Ready to generate.")


In [ ]:
%%writefile /content/generate_sentence_bank_audio.py
# ============================================================
# Sentence Bank TTS Generation Script
# Coqui XTTS v2 | Speaker: Ana Florence
# English → language='en' | Urdu → language='hi' (Hindi mode)
# ============================================================
import os
import re
import json
import torch
import numpy as np
import librosa
import soundfile as sf
from datetime import datetime
from TTS.api import TTS
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s — %(levelname)s — %(message)s')
logger = logging.getLogger('SentenceBankTTS')

os.environ['COQUI_TOS_AGREED'] = '1'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f'Device: {device}')

# ============================================================
# SPEAKER & MODEL SETTINGS (same as original TTS generator)
# ============================================================
SELECTED_SPEAKER = 'Ana Florence'

TTS_PARAMS = dict(
    temperature=0.65,        # Conservative — prevents random artifacts
    top_k=150,               # Focused vocabulary
    top_p=0.80,              # Less divergence
    repetition_penalty=1.5,  # Prevents repeated artifacts
    speed=0.95,              # Slightly slower for clearer enunciation
)

ENGLISH_FILES = [
    '1_blocks.json',
    '2_prolongation.json',
    '3_repetitions.json',
    '4_velar_fronting.json',
    '5_stopping.json',
    '6_gliding.json',
    '7_cluster_reduction.json',
    '8_epenthesis.json',
]

URDU_FILES = [
    'urdu_1_blocks.json',
    'urdu_2_prolongation.json',
    'urdu_3_repetitions.json',
    'urdu_4_velar_fronting.json',
    'urdu_5_stopping.json',
    'urdu_6_gliding.json',
    'urdu_7_cluster_reduction.json',
    'urdu_8_epenthesis.json',
]

# ============================================================
# TEXT PREPROCESSING
# ============================================================
def preprocess_english_text(text):
    # Collapse whitespace
    text = ' '.join(text.split())
    text = text.replace('...', '.').replace('..', '.')
    text = text.strip()
    # Hard stop prevents trailing hallucinations
    if text and text[-1] not in '.!?':
        text += '.'
    return text

def preprocess_urdu_text(text):
    # Collapse whitespace
    text = ' '.join(text.split())
    text = text.strip('\u06d4!\u061f ')
    # Space after punctuation for clarity
    text = text.replace('\u06d4', '\u06d4 ')
    text = text.replace('\u061f', '\u061f ')
    text = ' '.join(text.split())
    # Hard stop (Urdu danda ۔) prevents trailing hallucinations in Hindi/Urdu mode
    if text and text[-1] not in '\u06d4!\u061f':
        text += '\u06d4'
    return text

# ============================================================
# AUDIO POST-PROCESSING (identical to original TTS generator)
# ============================================================
def remove_leading_trailing_artifacts(file_path, trim_ms=250):
    try:
        y, sr = librosa.load(file_path, sr=None)
        trim_samples = int(sr * trim_ms / 1000)
        threshold = 0.01

        start_idx = 0
        for i in range(0, len(y) - trim_samples):
            if np.max(np.abs(y[i:i + trim_samples])) > threshold:
                start_idx = i
                break

        end_idx = len(y)
        for i in range(len(y) - 1, trim_samples, -1):
            if np.max(np.abs(y[i - trim_samples:i])) > threshold:
                end_idx = i
                break

        y_trimmed = y[start_idx:end_idx]
        if len(y_trimmed) > sr * 0.5:  # keep only if > 500 ms remains
            fade_samples = int(sr * 0.05)  # 50 ms fade in/out
            if fade_samples > 0 and len(y_trimmed) > fade_samples * 2:
                y_trimmed[:fade_samples]  *= np.linspace(0, 1, fade_samples)
                y_trimmed[-fade_samples:] *= np.linspace(1, 0, fade_samples)
            sf.write(file_path, y_trimmed, sr)
            return True
    except Exception as e:
        logger.warning(f'  ⚠️  Artifact removal failed: {e}')
    return False

def enhance_and_validate_audio(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None)
        peak_amplitude = np.max(np.abs(y))

        # Reject clipped audio (same threshold as original)
        if peak_amplitude > 0.99:
            logger.warning(f'    ⚠️  Clipping detected ({peak_amplitude:.2f}) — skipping')
            return False

        # Normalize to -1 dB
        if peak_amplitude > 0:
            target_linear = 10 ** (-1.0 / 20.0)
            y = y * (target_linear / peak_amplitude)

        # 80 Hz high-pass filter (removes rumble / low-freq noise)
        from scipy import signal
        sos = signal.butter(4, 80, 'hp', fs=sr, output='sos')
        y = signal.sosfilt(sos, y)

        # Final normalise
        max_val = np.max(np.abs(y))
        if max_val > 0:
            y = y / (max_val * 1.02)

        sf.write(file_path, y, sr)
        return True
    except Exception as e:
        logger.warning(f'  ⚠️  Audio enhancement failed: {e}')
        return False

# ============================================================
# SAFE FILENAME HELPER
# ============================================================
def safe_filename(text):
    # Keep Unicode word chars and hyphens; replace everything else with _
    cleaned = re.sub(r'[^\w\-]', '_', text, flags=re.UNICODE)
    cleaned = re.sub(r'_+', '_', cleaned).strip('_')
    return cleaned[:40]  # cap to avoid OS path limits

# ============================================================
# SINGLE-CLIP GENERATION
# ============================================================
def generate_clip(tts, text, out_path, lang_code, label):
    # Skip if already generated (resumable)
    if os.path.exists(out_path) and os.path.getsize(out_path) > 5000:
        logger.info(f'  ⏭️  {label} — already exists, skipping')
        return True

    try:
        tts.tts_to_file(
            text=text,
            file_path=out_path,
            speaker=SELECTED_SPEAKER,
            language=lang_code,
            **TTS_PARAMS,
        )
        remove_leading_trailing_artifacts(out_path, trim_ms=250)
        if enhance_and_validate_audio(out_path):
            logger.info(f'  ✅ {label} → saved')
            return True
        else:
            logger.warning(f'  ⚠️  {label} → quality check failed')
            return False
    except Exception as e:
        logger.error(f'  ❌ {label} failed: {e}')
        return False

# ============================================================
# PROCESS ONE SENTENCE BANK FILE
# ============================================================
def process_bank_file(tts, json_path, lang_code, language_dir, manifest_bank):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    test_type = data['test_type']
    sentences  = data['sentences']
    preprocess = preprocess_urdu_text if lang_code == 'hi' else preprocess_english_text

    sent_dir = os.path.join('audio_assets', language_dir, test_type, 'sentence')
    word_dir  = os.path.join('audio_assets', language_dir, test_type, 'word')
    os.makedirs(sent_dir, exist_ok=True)
    os.makedirs(word_dir,  exist_ok=True)

    logger.info(f'\n📂 [{language_dir.upper()} / {test_type}]  {len(sentences)} sentences')
    saved = failed = 0

    for sent in sentences:
        sid      = sent['id']
        raw_text = sent['text']
        targets  = sent.get('targets', [])

        # Sentence-level clip
        sent_file = f'{sid}.wav'
        sent_path = os.path.join(sent_dir, sent_file)
        sent_rel  = f'{language_dir}/{test_type}/sentence/{sent_file}'

        ok = generate_clip(tts, preprocess(raw_text), sent_path, lang_code, f'SENT {sid}')
        saved += ok
        failed += (not ok)

        # Word-level clips
        word_entries = []
        for idx, target in enumerate(targets):
            word      = target['word']
            word_fn   = f'{sid}_w{idx}_{safe_filename(word)}.wav'
            word_path = os.path.join(word_dir, word_fn)
            word_rel  = f'{language_dir}/{test_type}/word/{word_fn}'

            ok = generate_clip(tts, preprocess(word), word_path, lang_code,
                               f"WORD {sid}_w{idx} '{word}'")
            saved += ok
            failed += (not ok)

            word_entries.append({
                'index':        idx,
                'word':         word,
                'file':         word_rel,
                'expected_ipa': target.get('expected_ipa', []),
            })

        manifest_bank.append({
            'id':         sid,
            'text':       raw_text,
            'difficulty': sent.get('difficulty'),
            'file':       sent_rel,
            'words':      word_entries,
        })

    logger.info(f'  📊 [{test_type}] saved={saved}  failed={failed}')
    return saved, failed

# ============================================================
# MAIN
# ============================================================
logger.info('\nLoading Coqui XTTS v2...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
logger.info(f'🎤 Speaker : {SELECTED_SPEAKER}')
logger.info(f'⚙️  Params  : {TTS_PARAMS}')

manifest = {
    'version':      '1.0.0',
    'generated_at': datetime.now().isoformat(),
    'english':      {},
    'urdu':         {},
}

grand_saved = grand_failed = 0
start_time  = datetime.now()

# English banks
logger.info('\n' + '='*60)
logger.info('ENGLISH BANKS  (language=en)')
logger.info('='*60)
for fname in ENGLISH_FILES:
    path = f'/content/{fname}'
    if not os.path.exists(path):
        logger.error(f'❌ Not found: {fname} — skipping')
        continue
    with open(path, encoding='utf-8') as f:
        test_type = json.load(f)['test_type']
    bank_manifest = []
    s, fl = process_bank_file(tts, path, 'en', 'english', bank_manifest)
    manifest['english'][test_type] = bank_manifest
    grand_saved  += s
    grand_failed += fl

# Urdu banks (Hindi mode of XTTS v2)
logger.info('\n' + '='*60)
logger.info('URDU BANKS  (language=hi — XTTS v2 Hindi mode)')
logger.info('='*60)
for fname in URDU_FILES:
    path = f'/content/{fname}'
    if not os.path.exists(path):
        logger.error(f'❌ Not found: {fname} — skipping')
        continue
    with open(path, encoding='utf-8') as f:
        test_type = json.load(f)['test_type']
    bank_manifest = []
    s, fl = process_bank_file(tts, path, 'hi', 'urdu', bank_manifest)
    manifest['urdu'][test_type] = bank_manifest
    grand_saved  += s
    grand_failed += fl

# Save manifest
os.makedirs('audio_assets', exist_ok=True)
manifest_path = 'audio_assets/manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

elapsed = (datetime.now() - start_time).total_seconds()
logger.info('\n' + '='*60)
logger.info('✅ ALL DONE')
logger.info(f'   Clips saved  : {grand_saved}')
logger.info(f'   Clips failed : {grand_failed}')
logger.info(f'   Elapsed time : {elapsed/60:.1f} min')
logger.info(f'   Manifest     : {manifest_path}')
logger.info('='*60)


In [ ]:
# ============================================================
# STEP 4: Run TTS Generation
# ============================================================
# Estimated time:
#   16 banks × 45 sentences × ~3-4 target words each
#   ≈ ~2,880 clips @ ~6s each ≈ ~290 GPU minutes on T4
#   Resumable: re-running this cell skips already-generated files

print("4️⃣ Starting TTS generation...")
print("   ⏱️  Estimated: ~4-5 hours on T4 GPU")
print("   ♻️  Resumable: re-run this cell to skip already-generated clips")
print()

!cd /content && /content/tts-env/bin/python generate_sentence_bank_audio.py


In [ ]:
# ============================================================
# STEP 5: Inspect Manifest & File Counts
# ============================================================
import json, subprocess, os

result = subprocess.run(
    ['find', 'audio_assets', '-name', '*.wav'],
    capture_output=True, text=True, cwd='/content'
)
wav_files = [l for l in result.stdout.strip().split('\n') if l]

print(f'📊 Total WAV files : {len(wav_files)}')
print()

# FIX: find returns relative paths like 'audio_assets/english/...' (no leading slash)
for lang in ['english', 'urdu']:
    lang_wavs = [w for w in wav_files if f'audio_assets/{lang}/' in w]
    sent_wavs = [w for w in lang_wavs if '/sentence/' in w]
    word_wavs = [w for w in lang_wavs if '/word/' in w]
    print(f'  {lang.upper():8s}  sentence={len(sent_wavs):4d}  word={len(word_wavs):4d}  total={len(lang_wavs):4d}')

manifest_path = '/content/audio_assets/manifest.json'
if os.path.exists(manifest_path):
    with open(manifest_path, encoding='utf-8') as f:
        m = json.load(f)
    print()
    print(f"📋 Manifest     : {manifest_path}  ({os.path.getsize(manifest_path)/1024:.1f} KB)")
    print(f"   Generated    : {m.get('generated_at', 'N/A')}")
    print(f"   English types: {list(m.get('english', {}).keys())}")
    print(f"   Urdu types   : {list(m.get('urdu', {}).keys())}")

    print()
    print("📝 Sample manifest entry (first English sentence in 'blocks'):")
    try:
        sample = m['english']['blocks'][0]
        print(json.dumps(sample, indent=4, ensure_ascii=False))
    except (KeyError, IndexError):
        print('   (not available yet)')


In [ ]:
# ============================================================
# STEP 6: Zip & Download
# ============================================================
import shutil, os

print("5️⃣ Zipping audio assets...")
shutil.make_archive('/content/sentence_bank_audio_assets', 'zip', '/content/audio_assets')

zip_size = os.path.getsize('/content/sentence_bank_audio_assets.zip') / 1024 / 1024
print(f'   📦 ZIP size: {zip_size:.1f} MB')
print()
print('📁 Output structure inside ZIP:')
print('   manifest.json')
print('   english/')
print('     blocks/sentence/   en_blo_01.wav, ...')
print('     blocks/word/       en_blo_01_w0_Go.wav, ...')
print('     prolongation/ repetitions/ velar_fronting/')
print('     stopping/ gliding/ cluster_reduction/ epenthesis/')
print('   urdu/')
print('     blocks/sentence/   ur_blo_01.wav, ...')
print('     blocks/word/       ur_blo_01_w0_<word>.wav, ...')
print('     (+ 7 more test types)')
print()

from google.colab import files
print("6️⃣ Downloading sentence_bank_audio_assets.zip...")
files.download('/content/sentence_bank_audio_assets.zip')
print('\n🎉 Done!')


In [ ]:
# ============================================================
# OPTIONAL: Audio Quality Spot-Check
# ============================================================
import librosa, numpy as np, glob, os

def analyze_audio(wav_path):
    try:
        y, sr = librosa.load(wav_path, sr=None)
        duration = librosa.get_duration(y=y, sr=sr)
        peak     = np.max(np.abs(y))
        clipping = '⚠️  Yes' if peak >= 0.98 else '✅ No'
        return dict(duration_ms=duration * 1000, peak=peak, clipping=clipping)
    except:
        return None

print('📊 Audio Quality Spot-Check\n')
for lang in ['english', 'urdu']:
    pattern = f'/content/audio_assets/{lang}/*/sentence/*.wav'
    for wav in sorted(glob.glob(pattern))[:3]:
        m = analyze_audio(wav)
        if m:
            fname = os.path.relpath(wav, '/content/audio_assets')
            print(f'  {fname}')
            print(f'    Duration : {m["duration_ms"]:.0f} ms')
            print(f'    Peak Amp : {m["peak"]:.3f}')
            print(f'    Clipping : {m["clipping"]}')
            print()
